In [8]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import pandas as pd
from PyGol_Tabular import *

In [9]:
data_set = load_breast_cancer()
df = pd.read_csv("diabetes.csv")
target = "outcome"
X = df.drop(columns=[target])

y = df[target]
X_train, X_test, y_train, y_test = train_test_split(
        X,y,
        test_size=0.2, random_state=42, shuffle=True
    )

In [10]:
clf_bin = PyGolMultiClassifier(binner="entropy", max_literals=2, exact_literals=True, min_pos=1, max_neg = 0, n_bins=5, verbose=True)
clf_bin.fit(X_train, y_train)
y_pred_test = clf_bin.predict(X_test)
print("\nAcccuracy Score")
print(accuracy_score(y_test, y_pred_test))
print("\nTest Classification Report")
print(classification_report (y_test, y_pred_test))
print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_test))


Acccuracy Score
0.8975

Test Classification Report
              precision    recall  f1-score   support

           0       0.87      0.99      0.92       253
           1       0.98      0.73      0.84       147

    accuracy                           0.90       400
   macro avg       0.92      0.86      0.88       400
weighted avg       0.91      0.90      0.89       400


Confusion Matrix
[[251   2]
 [ 39 108]]


In [13]:
cf = PyGolCounterfactual(clf_bin, X_train)

# Pick any row index — flip happens automatically
result = cf.flip_example(
    X_test,
    row_index   = 0,
    class_names = {0: "Non-diabetic", 1: "Diabetic"},
    target_class = None,   # None = auto-flip to opposite
    verify      = True,
)


════════════════════════════════════════════════════════════════════
  flip_example  ·  row 0
════════════════════════════════════════════════════════════════════

  PART 1 · Why this example is predicted as  Diabetic (class 1)
  ────────────────────────────────────────────────────────────────────
  2 rule(s) currently fire for class  Diabetic (class 1):

  Rule  1:  IF  pregnancies = 4.0
          AND diabetespedigreefunction = [1.275,1.397]
           └─ pregnancies                           current = 4  bin = 4.0
           └─ diabetespedigreefunction              current = 1.39  bin = [1.275,1.397]

  Rule  2:  IF  pregnancies = 4.0
          AND age = [43.5,61.5]
           └─ pregnancies                           current = 4  bin = 4.0
           └─ age                                   current = 56  bin = [43.5,61.5]

  PART 2 · Counterfactual — changes to flip to  Non-diabetic (class 0)
  ────────────────────────────────────────────────────────────────────
  2 change(s) needed

In [12]:
print("Number of Changes Made:", result.n_changes)
print("Number of Target Rules Fired after CF:", len(result.target_rules_fired))


Number of Changes Made: 1
Number of Target Rules Fired after CF: 4
